# Profile plots

Plots that need only the small `_profiles.nc` files (`plots/fetch_profiles.sh` copies them from the cluster). Surface, 3D and checkpoint plots are in `plots.ipynb`.

`read_profiles` (in `plotter.jl`, `?read_profiles`) returns arrays; compute with them here (`1e3 .* q`, `A .- A[:, 1:1]`, `log10.(max.(q, 1e-6))`, `mean(A; dims = 2)`) and plot with `Figure`, `Axis`, `heatmap!`/`lines!`, `Colorbar`. Variable names: `plots/fields.md`.

In [11]:
# plots/ is the folder with plotter.jl: look from where the kernel started, then at the usual checkouts (hpc, Mac)
plotsdir = first(filter( d -> isfile(joinpath(d, "plotter.jl")) ,
            ( pwd(), joinpath(pwd(), "plots"), dirname(pwd()),
              "/ceoas/deszoeks/projects/monsoon-breeze/plots",
              expanduser("~/projects/monsoon-breeze/plots") )   ))
cd(plotsdir)
using Pkg; Pkg.activate(plotsdir)
using Revise
includet(joinpath(plotsdir, "plotter.jl"))

rundir = joinpath(dirname(plotsdir), "run", "ekamsat-p3-ctl0")

  Activating project at `/ceoas/deszoeks/projects/monsoon-breeze/plots`


"/ceoas/deszoeks/projects/monsoon-breeze/run/ekamsat-p3-ctl0"

In [ ]:
p = read_profiles(rundir)       # NamedTuple: p.t (days), p.z and p.zᶠ (m), each variable as (z, time)
keys(p)

In [ ]:
# One time–height panel in fig[i, j]; A is (z, time) at heights z (m). Returns the axis and heatmap.
function th_panel!(fig, i, j, t, z, A; title, colormap = :dense, colorrange = finite_range(A), zmax = 12)
    ax = Axis(fig[i, j]; title, ylabel = "z (km)", limits = (nothing, (0, zmax)))
    hm = heatmap!(ax, t, z ./ 1e3, A'; colormap, colorrange)
    return ax, hm
end

# (title, array, colormap, color range function): anomalies from the first hour, condensates in g/kg
panels = [("T (K), change since first hour", p.T .- p.T[:, 1:1], :balance, symmetric_range),
          ("qv (g/kg), change", 1e3 .* (p.qv .- p.qv[:, 1:1]), :balance, symmetric_range),
          ("qcl (g/kg)", 1e3 .* p.qcl, :dense, finite_range),
          ("qi (g/kg)", 1e3 .* p.qi, :dense, finite_range),
          ("qr (g/kg)", 1e3 .* p.qr, :dense, finite_range),
          ("qf (g/kg), rime", 1e3 .* p.qf, :dense, finite_range),
          ("radiative flux divergence (W/m³)", p.radiative_flux_divergence, :balance, symmetric_range)]
fig = Figure(size = (900, 1700))
axs = Axis[]
for (i, (title, A, colormap, rng)) in enumerate(panels)
    ax, hm = th_panel!(fig, i, 1, p.t, p.z, A; title, colormap, colorrange = rng(A))
    contour!(ax, p.t, p.z ./ 1e3, p.T'; levels = [273.15], color = :black, linestyle = :dash)   # 0 °C
    Colorbar(fig[i, 2], hm)
    push!(axs, ax)
end
foreach(ax -> hidexdecorations!(ax; grid = false), axs[1:end-1])
axs[end].xlabel = "time (days)"
linkxaxes!(axs...)
fig

In [ ]:
# Zoom: set the limits of the axes from the previous cell, then show the figure again.
for ax in axs
    xlims!(ax, 1, 2)      # days
    ylims!(ax, 0, 5)      # km
end
fig

In [ ]:
# Wind: u, v, and the rms vertical velocity √⟨w²⟩ (on z faces)
winds = (("u (m/s)", p.u, p.z, :balance, symmetric_range),
         ("v (m/s)", p.v, p.z, :balance, symmetric_range),
         ("√⟨w²⟩ (m/s)", sqrt.(max.(p.w2, 0)), p.zᶠ, :dense, finite_range))
fig_wind = Figure(size = (900, 750))
for (i, (title, A, z, colormap, rng)) in enumerate(winds)
    ax, hm = th_panel!(fig_wind, i, 1, p.t, z, A; title, colormap, colorrange = rng(A), zmax = 20)
    Colorbar(fig_wind[i, 2], hm)
end
fig_wind

In [ ]:
# Profiles at the first and last hour
profs = (("θ (K)", p.theta), ("qv (g/kg)", 1e3 .* p.qv), ("qcl + qi (g/kg)", 1e3 .* (p.qcl .+ p.qi)),
         ("u (m/s)", p.u), ("v (m/s)", p.v))
fig_profiles = Figure(size = (1500, 600))
for (j, (xlabel, A)) in enumerate(profs)
    ax = Axis(fig_profiles[1, j]; xlabel, ylabel = j == 1 ? "z (km)" : "", limits = (nothing, (0, 16)),
              xticks = WilkinsonTicks(3))
    j > 1 && hideydecorations!(ax; grid = false)
    for n in (1, size(A, 2))
        lines!(ax, A[:, n], p.z ./ 1e3; label = "$(round(p.t[n]; digits = 2)) d")
    end
end
Legend(fig_profiles[1, length(profs) + 1], content(fig_profiles[1, 1]))
fig_profiles

## P3 runs

Fetch the small profile files on the Mac with `plots/fetch_profiles.sh`; on the cluster they are already there. Control extension and the three moistening branches, from the 2-day branch point.

In [ ]:
p3runs = ["ctl2", "moisten1000-850", "moisten850-650", "moisten650-440"]
p3dir(r) = joinpath("../run", "ekamsat-p3-$r")
p3 = [read_profiles(p3dir(r)) for r in p3runs]      # one NamedTuple per run

In [ ]:
# Runs in columns, variables in rows; one color scale and colorbar per row (th_panel! is defined above)
rows = [("qv (g/kg), change since first hour", q -> 1e3 .* (q.qv .- q.qv[:, 1:1]), :balance, symmetric_range),
        ("T (K), change", q -> q.T .- q.T[:, 1:1], :balance, symmetric_range),
        ("qcl (g/kg)", q -> 1e3 .* q.qcl, :dense, finite_range),
        ("qi (g/kg)", q -> 1e3 .* q.qi, :dense, finite_range),
        ("qr (g/kg)", q -> 1e3 .* q.qr, :dense, finite_range),
        ("qf (g/kg), rime", q -> 1e3 .* q.qf, :dense, finite_range)]
fig_p3 = Figure(size = (1500, 1500))
for (i, (label, getA, colormap, rng)) in enumerate(rows)
    As = getA.(p3)
    colorrange = rng(reduce(vcat, vec.(As)))
    hm = nothing
    for (j, r) in enumerate(p3runs)
        ax, hm = th_panel!(fig_p3, i, j, p3[j].t, p3[j].z, As[j]; title = i == 1 ? r : "", colormap, colorrange, zmax = 16)
        j > 1 && hideydecorations!(ax; grid = false)
        i < length(rows) ? hidexdecorations!(ax; grid = false) : (ax.xlabel = "time (days)")
    end
    Colorbar(fig_p3[i, length(p3runs) + 1], hm; label)
end
fig_p3

In [ ]:
# Profiles at the last hour, one line per run
vars = (("θ (K)", :theta, 1), ("qv (g/kg)", :qv, 1e3), ("qi (g/kg)", :qi, 1e3), ("qr (g/kg)", :qr, 1e3))
fig_p3_profiles = Figure(size = (1200, 600))
for (j, (xlabel, name, scale)) in enumerate(vars)
    ax = Axis(fig_p3_profiles[1, j]; xlabel, ylabel = j == 1 ? "z (km)" : "", limits = (nothing, (0, 16)),
              xticks = WilkinsonTicks(3))
    j > 1 && hideydecorations!(ax; grid = false)
    for (r, q) in zip(p3runs, p3)
        lines!(ax, scale .* getproperty(q, name)[:, end], q.z ./ 1e3; label = r)
    end
end
Legend(fig_p3_profiles[1, length(vars) + 1], content(fig_p3_profiles[1, 1]))
fig_p3_profiles

In [ ]:
# Arrays can be combined before plotting: qv of each moistening run minus ctl2 (g/kg), on the output times all runs have
n = minimum(size(q.qv, 2) for q in p3)
Δqv = [1e3 .* (q.qv[:, 1:n] .- p3[1].qv[:, 1:n]) for q in p3[2:end]]
colorrange = symmetric_range(reduce(vcat, vec.(Δqv)))
fig_diff = Figure(size = (1200, 330))
for (j, A) in enumerate(Δqv)
    ax, hm = th_panel!(fig_diff, 1, j, p3[1].t[1:n], p3[1].z, A; title = p3runs[j + 1], colormap = :balance, colorrange, zmax = 16)
    j > 1 && hideydecorations!(ax; grid = false)
    j == length(Δqv) && Colorbar(fig_diff[1, j + 1], hm; label = "qv − qv(ctl2) (g/kg)")
end
fig_diff

In [20]:
# Numbers behind the P3 first look in notes/STATUS.md: changes from the first to the last output
function p3_summary(r)
    p = read_profiles(p3dir(r))
    zk = p.z ./ 1e3
    Δqv = 1e3 .* (p.qv[:, end] .- p.qv[:, 1])             # g/kg
    ΔT = p.T[:, end] .- p.T[:, 1]                          # K
    top = 10 .<= zk .<= 12
    freezing = zk[findlast(p.T[:, end] .> 273.15)]         # top of the T > 0 °C layer of the mean
    rain_top = zk[findlast(p.qr[:, end] .> 1e-6)]          # highest rain above 0.001 g/kg
    (run = r, days = round(p.t[end]; digits = 2),
     Δqv_max = round(maximum(Δqv); digits = 2), Δqv_z = zk[argmax(Δqv)],
     ΔT_max = round(maximum(ΔT); digits = 2), ΔT_min = round(minimum(ΔT); digits = 2),
     ice_max = round(1e3 * maximum(p.qi[:, end]); digits = 3),
     rime_over_ice = round(mean(p.qf[top, end]) / mean(p.qi[top, end]); digits = 2),   # 10–12 km
     freezing_km = freezing, rain_top_km = rain_top)
end
p3_summary.(p3runs)

4-element Vector{@NamedTuple{run::String, days::Float64, Δqv_max::Float64, Δqv_z::Float64, ΔT_max::Float32, ΔT_min::Float32, ice_max::Float64, rime_over_ice::Float32, freezing_km::Float64, rain_top_km::Float64}}:
 (run = "ctl2", days = 5.71, Δqv_max = 1.09, Δqv_z = 6.25, ΔT_max = 6.6, ΔT_min = -0.88, ice_max = 0.065, rime_over_ice = 0.46, freezing_km = 4.761841796875, rain_top_km = 7.25)
 (run = "moisten1000-850", days = 5.58, Δqv_max = 1.56, Δqv_z = 0.17236842346191406, ΔT_max = 6.87, ΔT_min = -0.82, ice_max = 0.088, rime_over_ice = 0.45, freezing_km = 4.761841796875, rain_top_km = 7.75)
 (run = "moisten850-650", days = 3.88, Δqv_max = 1.38, Δqv_z = 2.007894775390625, ΔT_max = 3.81, ΔT_min = -0.81, ice_max = 0.064, rime_over_ice = 0.47, freezing_km = 4.761841796875, rain_top_km = 7.25)
 (run = "moisten650-440", days = 3.96, Δqv_max = 2.24, Δqv_z = 4.761841796875, ΔT_max = 4.14, ΔT_min = -0.72, ice_max = 0.089, rime_over_ice = 0.48, freezing_km = 4.761841796875, rain_top_km = 7.75)